# Crossing Watch: the tuned Glimmer judge (LoRA) on Kaggle, for $0

The LoRA-tuned Muse Glimmer judge (`provenance/glimmer-judge-lora.json`: rank 8, alpha 32, trained on Fireworks'
Serverless Training API) runs here on Kaggle's free **2 x T4**, on the same Meta 4-bit base and the same llama.cpp
`b11976` as `<removed-run-record>`, and judges the public planted-lies bench (`kaggle/judge-bench-public.json`,
letters about the synthetic `sample/` only) once per letter at temperature 0, as the training session scored it.

**Settings → Accelerator: GPU T4 x2. Internet: on.** Two inputs, both with **Add Input**:

1. `crossing-watch`: a zip of one commit of the repo, made with
   `git archive --format=zip --add-virtual-file=GIT_SHA:$(git rev-parse HEAD) -o crossing-watch.zip HEAD`
2. a **private** dataset holding the adapter exactly as Fireworks gives it (`adapter_config.json` and
   `adapter_model.safetensors`), downloaded with `firectl model download
   accounts/<fireworks-account>/models/muse-starter-sl-judge-sl-20261031 <dir>` after `firectl signin` (an API key
   cannot download models).

Steps: build llama-server; fetch the pinned base GGUF (sha256 checked) and the base's `config.json` (pinned revision,
config only); convert the adapter to GGUF with llama.cpp's `convert_lora_to_gguf.py`, after a one-line fix (its
`LoraTorchTensor` has no `ndim`, which Glimmer's q/k unpermute reads); start llama-server with `--lora` and the
training template (`Reasoning strength: high.`); check `/apply-template` against the training renderer's header; run
the bench; write `glimmer-judge-lora-provenance.json` (adapter sha256, llama.cpp build, template check, timings).
Weights trained in BF16 run here over a Q4_K_M base: expect a small drift from the in-session numbers.

In [ ]:
import json, subprocess, time
t0 = time.time()  # the provenance file gives each step's end in seconds from here
NVIDIA_SMI = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=True).stdout
GPUS = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True,
                      check=True).stdout.splitlines()
HOST = f"Kaggle {len(GPUS)} x {GPUS[0].removeprefix('Tesla ')}"
print(NVIDIA_SMI, HOST)

## 1. Build llama.cpp's server for the T4 (about 28 minutes), and fix the LoRA converter

In [ ]:
%%bash
set -eo pipefail
cd /kaggle/working
[ -d llama.cpp ] || git clone --depth 1 --branch b11976 https://github.com/ggml-org/llama.cpp
cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=ON -DGGML_CUDA_NO_VMM=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF | tail -3
cmake --build llama.cpp/build --config Release -j 4 --target llama-server 2>&1 | tail -2
# the fix: conversion/muse_glimmer.py unpermutes q/k with tensor.ndim, which LoraTorchTensor lacks (b11976 and master)
F=llama.cpp/convert_lora_to_gguf.py
grep -q 'def ndim' $F || sed -i 's/^    def dim(self) -> int:$/    @property\n    def ndim(self) -> int:\n        return len(self.shape)\n\n&/' $F
[ "$(grep -c 'def ndim' $F)" = 1 ] && python3 -c "import ast, sys; ast.parse(open(sys.argv[1]).read())" $F && echo 'ndim fix applied'

## 2. The base (pinned, sha256 checked), its config, and the adapter (sha256, config, tensor names)

In [ ]:
import glob, hashlib, math, os, shutil, struct
from huggingface_hub import hf_hub_download
marks = {'built': round(time.time() - t0)}
REPO, REVISION = 'meta-models/Muse-Glimmer-30B-GGUF', 'b44f1072cf75e918a1b82a3a17b45c4389b076e7'
BASE_GGUF = 'Muse-Glimmer-30B-KQuant-17GB-Q4_K_M.gguf'
BASE_SHA256 = '4cc57c0f51040a226e5a72cc47b7613f7772950e460a665f7083de89f183f60e'
CONFIG_REPO, CONFIG_REVISION = 'meta-models/Muse-Glimmer-30B', 'cec94fcfd6512f2cfe708138c32533b346ea4f69'


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()


model = hf_hub_download(REPO, BASE_GGUF, revision=REVISION, local_dir='/tmp/glimmer')
assert sha256(model) == BASE_SHA256, 'the base is not the file Hugging Face published'
BASE_DIR = '/kaggle/working/base-config'  # config.json only: the converter needs no weights
hf_hub_download(CONFIG_REPO, 'config.json', revision=CONFIG_REVISION, local_dir=BASE_DIR)

src = os.path.dirname(glob.glob('/kaggle/input/**/pack.py', recursive=True)[0])
CW = '/kaggle/working/cw'
shutil.copytree(src, CW, dirs_exist_ok=True)
TRAINED = json.load(open(f'{CW}/provenance/glimmer-judge-lora.json'))['params']  # rank 8, alpha 32

found = glob.glob('/kaggle/input/**/adapter_model.safetensors', recursive=True)
if len(found) != 1:
    raise RuntimeError(f'add the private adapter dataset as an input (adapter_model.safetensors found: {found})')
LORA = '/kaggle/working/lora'  # a writable copy: the converter reads adapter_config.json beside the weights
shutil.copytree(os.path.dirname(found[0]), LORA, dirs_exist_ok=True)
ADAPTER_SHA256 = {name: sha256(f'{LORA}/{name}') for name in sorted(os.listdir(LORA)) if os.path.isfile(f'{LORA}/{name}')}
config_path = f'{LORA}/adapter_config.json'
adapter_config = json.load(open(config_path)) if os.path.exists(config_path) else {}
config_written = not {'r', 'lora_alpha'} <= adapter_config.keys()
if config_written:  # only the training record's own values, never a guess
    adapter_config = {'r': TRAINED['lora_rank'], 'lora_alpha': TRAINED['lora_alpha'], **adapter_config}
    json.dump(adapter_config, open(config_path, 'w'), indent=1)
assert (adapter_config['r'], adapter_config['lora_alpha']) == (TRAINED['lora_rank'], TRAINED['lora_alpha']), adapter_config

with open(f'{LORA}/adapter_model.safetensors', 'rb') as f:  # safetensors: 8-byte length, then a JSON header
    header = json.loads(f.read(struct.unpack('<Q', f.read(8))[0]))
tensors = {k: v['shape'] for k, v in header.items() if k != '__metadata__'}
ADAPTER = {'files_sha256': ADAPTER_SHA256, 'config_written_by_notebook': config_written,
           'r': adapter_config['r'], 'lora_alpha': adapter_config['lora_alpha'],
           'target_modules': adapter_config.get('target_modules'), 'tensors': len(tensors),
           'params': sum(math.prod(s) for s in tensors.values()),
           'lm_head_trained': any('lm_head' in k for k in tensors),
           'first_tensors': dict(list(tensors.items())[:4])}
marks['downloaded'] = round(time.time() - t0)
print(json.dumps(ADAPTER, indent=1))

## 3. Convert the adapter to GGUF (f16, about 108 MB)

In [ ]:
LORA_GGUF = '/kaggle/working/glimmer-judge-lora-f16.gguf'
conv = subprocess.run(['python3', '/kaggle/working/llama.cpp/convert_lora_to_gguf.py', LORA, '--base', BASE_DIR,
                       '--outtype', 'f16', '--outfile', LORA_GGUF], capture_output=True, text=True)
print(conv.stderr[-3000:])
if conv.returncode or not os.path.exists(LORA_GGUF):
    raise RuntimeError('convert_lora_to_gguf.py failed (output above)')
ADAPTER['gguf'] = {'sha256': sha256(LORA_GGUF), 'bytes': os.path.getsize(LORA_GGUF)}
marks['converted'] = round(time.time() - t0)
ADAPTER['gguf']

## 4. Start the server with `--lora` and the training template, and check the rendered prompt

Training rendered every row with the cookbook's Glimmer renderer: `Reasoning strength: high.` and the training day as
the current date (`provenance/glimmer-judge-lora.json`: Sat 31 Oct 2026). `/apply-template` must give the same text
as the renderer's own unit test (`test_muse_glimmer_renderer.py`) for the same message, or the bench is not run.

In [ ]:
import urllib.request
LOG = '/kaggle/working/server.log'
TEMPLATE_KWARGS = {'reasoning_strength': 'high', 'current_date': '2026-10-31'}  # as the training rows were rendered
ARGS = ['-m', model, '--lora', LORA_GGUF, '-ngl', '99', '--split-mode', 'layer', '--jinja', '--port', '8080',
        '-np', '1', '-c', '8192',  # a judge prompt is about 1.6k tokens and the tuned reply about 90 characters
        '--chat-template-kwargs', json.dumps(TEMPLATE_KWARGS),
        '--temp', '0']  # the starter's evaluate() sampled at temperature 0.0
server = subprocess.Popen(['/kaggle/working/llama.cpp/build/bin/llama-server', *ARGS], stdout=open(LOG, 'a'),
                          stderr=subprocess.STDOUT)
for _ in range(120):
    if server.poll() is not None:
        raise RuntimeError('llama-server exited:\n' + open(LOG).read()[-3000:])
    try:
        if urllib.request.urlopen('http://127.0.0.1:8080/health', timeout=5).status == 200:
            break
    except OSError:
        time.sleep(10)
else:
    raise RuntimeError('llama-server never got healthy:\n' + open(LOG).read()[-3000:])
PROPS = json.load(urllib.request.urlopen('http://127.0.0.1:8080/props', timeout=30))
LORAS = json.load(urllib.request.urlopen('http://127.0.0.1:8080/lora-adapters', timeout=30))
marks['server_ready'] = round(time.time() - t0)


def post(path, body):
    req = urllib.request.Request(f'http://127.0.0.1:8080{path}', json.dumps(body).encode(),
                                 {'Content-Type': 'application/json'})
    return json.load(urllib.request.urlopen(req, timeout=60))


EXPECTED = ('<|start|>system<|message|>You are a helpful AI assistant.\nKnowledge cutoff: 2026-01-04.\n'
            'Current date: 2026-10-31.\n\nReasoning strength: high.\n\n# Valid recipients: "self", "user".<|eot|>'
            '<|start|>user<|message|> hi <|eot|><|start|>assistant')
rendered = post('/apply-template', {'messages': [{'role': 'user', 'content': ' hi '}]})['prompt']
TEMPLATE_MATCH = rendered.removeprefix('<|begin_of_text|>') == EXPECTED
print('build:', PROPS.get('build_info'), '| lora:', LORAS, '| template matches training:', TEMPLATE_MATCH)
if not TEMPLATE_MATCH or not LORAS:
    raise RuntimeError(f'not the trained setup: lora {LORAS}, rendered {rendered!r}')

## 5. The tuned judge on the public bench, one vote a letter, and `glimmer-judge-lora-provenance.json`

Only the tuned Glimmer is asked (`JUDGE_MODEL_URL` is this server, the Meta key is left out); the bench's Spark
column is kept as committed for comparison. Download `glimmer-judge-lora-bench.txt`, `glimmer-judge-lora-bench.json`
and `glimmer-judge-lora-provenance.json` from the output (sample/ letters only: safe to publish) and commit them under
`kaggle/`.

In [ ]:
BENCH_OUT = '/kaggle/working/glimmer-judge-lora-bench.txt'
BENCH_JSON = '/kaggle/working/glimmer-judge-lora-bench.json'
bench_items = json.load(open(f'{CW}/kaggle/judge-bench-public.json'))
# any Glimmer verdict already in the file is base Glimmer's: drop it, or judge_bench.py would ask the tuned one nothing
bench_items = [{k: v for k, v in i.items() if k not in ('glimmer', 'glimmer_votes')} for i in bench_items]
json.dump(bench_items, open(BENCH_JSON, 'w'), indent=1, ensure_ascii=False)
bench_env = {**{k: v for k, v in os.environ.items() if k != 'MODEL_API_KEY'},
             'JUDGE_MODEL_URL': 'http://127.0.0.1:8080/v1', 'JUDGE_VOTES': '1'}
started = time.time()
bench = subprocess.run(['python3', 'judge_bench.py', '--out', BENCH_JSON, '--glimmer', '1'], cwd=CW, env=bench_env,
                       capture_output=True, text=True)
bench_s = round(time.time() - started)
replies = bench.stderr.count('open model reply:')
git_sha = open(f'{CW}/GIT_SHA').read().strip() if os.path.exists(f'{CW}/GIT_SHA') else None
with open(BENCH_OUT, 'w') as f:
    f.write(f'judge_bench.py: exit {bench.returncode}, {bench_s} s on {HOST}, {replies} replies from the tuned Glimmer '
            f'(LoRA {ADAPTER["gguf"]["sha256"][:12]}), JUDGE_VOTES=1, temp 0, git {git_sha}\n\n'
            f'{bench.stdout}\n{bench.stderr[-3000:]}')
print(open(BENCH_OUT).read())
marks['benched'] = round(time.time() - t0)

judged = json.load(open(BENCH_JSON))
flagged = lambda planted: sum(any(v for v in i.get('glimmer', []) if v) for i in judged if i['planted'] == planted)
provenance = {
    'notebook': 'kaggle/glimmer-judge-lora.ipynb',
    'promoted_model': 'accounts/<fireworks-account>/models/muse-starter-sl-judge-sl-20261031',
    'code': {'git_sha': git_sha},
    'nvidia_smi': NVIDIA_SMI,
    'adapter': ADAPTER,
    'base': {'repo': REPO, 'revision': REVISION, 'file': BASE_GGUF, 'sha256_checked': BASE_SHA256,
             'config': f'{CONFIG_REPO}@{CONFIG_REVISION}'},
    'llama_cpp': {'build_info': PROPS.get('build_info'), 'args': ARGS, 'lora_adapters': LORAS,
                  'converter_fix': 'LoraTorchTensor.ndim added (sed, cell 1)'},
    'template': {'kwargs': TEMPLATE_KWARGS, 'matches_training_renderer': TEMPLATE_MATCH},
    'bench': {'file': 'kaggle/judge-bench-public.json', 'exit': bench.returncode, 'seconds': bench_s, 'replies': replies,
              's_per_reply': round(bench_s / replies, 1) if replies else None,
              'lies_flagged': f"{flagged(True)}/{sum(i['planted'] for i in judged)}",
              'clean_flagged': f"{flagged(False)}/{sum(not i['planted'] for i in judged)}",
              'unreadable': sum(v is None for i in judged for v in i.get('glimmer', []))},
    'elapsed_s': marks,
}
with open('/kaggle/working/glimmer-judge-lora-provenance.json', 'w') as f:
    json.dump(provenance, f, indent=1, ensure_ascii=False)
print(json.dumps(provenance['bench'], indent=1))
if bench.returncode:
    raise RuntimeError(f'judge_bench.py failed; its output is in {BENCH_OUT}')